# Lab 02 - Model Architecture, Size, and Memory

**Chapter 2 (Understanding Foundation Models): Training Data, Modeling, Model Size** | Environment: **cluster** (your Docker JupyterLab) | Time: about 4 hours

At Larkspur Ridge IT you are the AI engineer building the help-desk assistant. Before you choose a model, you need to answer two practical questions the chapter keeps returning to: *how big is this model really*, and *what does that size cost me in memory and speed?* In Part A you read the real configuration files of the four course models and do the memory math by hand. In Part B you measure how fast three of them actually generate text on the course cluster and connect the speed to the number of **active** parameters.

**Before you start**
- Connect to the **KU VPN**, run `docker compose up`, and open this notebook in JupyterLab (http://localhost:8888).
- Your `notebooks/00_setup_check.ipynb` from Lab 01 should be all green.
- Use **Run > Run All Cells** once at the start to see where the TODOs are. Then work top to bottom.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Parameters (weights)** | The numbers a model learned during training (from Lab 01). "27B" means about 27 billion. | Memory and speed both start from how many there are. |
| **`config.json`** | A short text file published with every open-weight model. It lists the model's design settings: how many layers, how wide, how many experts. | You can work out a model's size and memory needs from it without downloading the model. |
| **Transformer block (layer)** | The repeated building block that is stacked to make a model. Each one has an attention part and an MLP part. | How many layers and how wide they are decides the parameter count. |
| **Attention** | The part of each layer where a token looks back at the earlier tokens to use their context. | Its memory use grows with the length of the conversation (see KV cache). |
| **MLP (feedforward layer)** | The part of each layer that processes each token on its own. It holds most of the parameters. | It's what you count in A.2. |
| **Dense model** | A model that uses every one of its parameters for every token. | Simple, but every token pays for the whole model. |
| **Mixture of experts (MoE)** | A model whose MLP is split into many small MLPs called **experts**. A small **router** picks a few experts for each token. | You get a big model's knowledge at a small model's cost per token. |
| **Total vs active parameters** | Total: everything the model stores. Active: the part actually used for one token. "35B-A3B" means 35B total, about 3B active. | Total decides how much memory you need; active decides how fast it runs. |
| **Precision, bf16** | How many bits store each number (8 bits = 1 byte). Models are usually published in **bf16** ("bfloat16"), a 16-bit format, so 2 bytes per parameter. | Fewer bits means less memory. |
| **Quantization** | Storing a model's weights with fewer bits (8 or 4 instead of 16), usually with a small loss in quality. | It's how the course cluster fits these models into memory. |
| **KV cache** | The keys and values (explained in A.4) that the model saves for every earlier token so it doesn't recompute them. | Memory that grows with every token of every conversation. |
| **Context length** | How many tokens are in one request: instructions, history, and the answer. | Longer context means a bigger KV cache. |
| **Memory bandwidth** | How many gigabytes per second a chip can read from its memory. | For a single request it usually sets the speed limit of text generation (Part B). |
| **Latency / tokens per second** | Latency: seconds from sending a request to getting the full reply. Tokens per second: how fast the model writes its answer. | This is what users feel as "fast" or "slow". |
| **Gateway, alias** | The **gateway** is the course server you send every request to; it passes each one to the right Mac Studio. An **alias** is the course's short name for a model, like `qwen-moe`. | Your code names only the alias, so the instructor can swap the model behind it without breaking your notebook. |
| **Scaling law, compute-optimal** | A rule found by research that predicts how good a model gets for a given amount of training compute, model size, and training data. "Compute-optimal" is the size and data mix that gives the best model for a fixed training budget. | Q4 asks why model makers often don't stop there. |

## Setup
Run this cell. It connects to the **course gateway** (the server on the KU network that passes your requests to the Mac Studios) and prints the models it serves, by their course names (**aliases**, like `qwen-moe`). It also creates your self-checker `ck`.

In [ ]:
import json, math, time, statistics
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt

from itec632 import get_client, MODELS, NO_THINK
from itec632.grading import LabChecker

ck = LabChecker("lab02", part="cluster")
client = get_client()

available = {m.id for m in client.models.list().data}
print("Models visible on the gateway:", sorted(available))
ck.check("setup_models_visible",
         {MODELS["main"], MODELS["dense"], MODELS["small"]} <= available, 1,
         "Connect to the KU VPN and re-run this cell. If it stays red, run notebooks/00_setup_check.ipynb.");

# Part A - Guided practice: reading a model by its config

Every open-weight model on Hugging Face (the website that hosts open models, from Lab 01) ships a `config.json`, a short text file listing the model's design settings. It is the blueprint the chapter describes in **"Modeling"**: how many transformer blocks, how wide each one is, how many attention heads, and (for a mixture-of-experts model) how many experts. With it you can estimate parameter counts and memory without downloading a single weight.

The course gateway serves four chat models. Their Hugging Face repositories are below. They are all newer than the models in the book, but the ideas are the same.

| Gateway alias | Hugging Face repo | Type |
|---|---|---|
| `qwen-moe` | `Qwen/Qwen3.6-35B-A3B` | Mixture of experts (MoE) |
| `qwen-dense` | `Qwen/Qwen3.8-27B` | Dense |
| `gemma-moe` | `google/gemma-4-26B-A4B-it` | MoE, different family |
| `qwen-small` | `Qwen/Qwen3.5-4B` | Dense, small |

## A.1 Download the configs

The next cell downloads each `config.json` over plain HTTPS. If the download fails (no internet, Hugging Face down), it falls back to a snapshot saved in this lab's `model_configs/` folder, so you can keep working either way. These models are multimodal, so the file has a `text_config` section for the language model; we use that part.

In [ ]:
REPOS = {
    "qwen-moe":   "Qwen/Qwen3.6-35B-A3B",
    "qwen-dense": "Qwen/Qwen3.8-27B",
    "gemma-moe":  "google/gemma-4-26B-A4B-it",
    "qwen-small": "Qwen/Qwen3.5-4B",
}
SNAPSHOT_DIR = Path("model_configs")   # bundled with this lab


def load_config(alias):
    """Return (text_config dict, source string) for one gateway alias."""
    url = f"https://huggingface.co/{REPOS[alias]}/resolve/main/config.json"
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        cfg, source = r.json(), "huggingface.co"
    except Exception as e:  # no internet, blocked, or Hugging Face down
        path = SNAPSHOT_DIR / f"{alias}.json"
        if not path.exists():
            raise FileNotFoundError(f"Download failed ({type(e).__name__}) and {path} is missing. "
                                    "Open this notebook from inside labs/lab02-model-size/.") from e
        cfg, source = json.loads(path.read_text()), f"snapshot {path}"
    return cfg.get("text_config", cfg), source


configs = {}
for alias in REPOS:
    configs[alias], src = load_config(alias)
    print(f"{alias:11} <- {src}")

FIELDS = ["hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim",
          "intermediate_size", "num_experts", "num_experts_per_tok", "top_k_experts",
          "moe_intermediate_size", "vocab_size", "tie_word_embeddings"]
table = pd.DataFrame({a: {f: c.get(f, "-") for f in FIELDS} for a, c in configs.items()})
for a, c in configs.items():
    lt = c.get("layer_types", [])
    table.loc["layer types", a] = ", ".join(f"{t}: {lt.count(t)}" for t in sorted(set(lt))) or "-"
table

### How to read this table

Each column is one model; each row is one setting from its `config.json`. A dash (`-`) means that model doesn't have that setting (for example, dense models have no experts).

First, a quick picture of what these numbers describe. A language model turns every token into a list of numbers, then passes that list through a **stack of identical layers** (transformer blocks). Each layer has two parts: **attention**, where each token looks back at the earlier tokens, and an **MLP** (feedforward) part, where each token is processed on its own. Almost all the parameters live in these layers, so the settings below are what decide the model's size.

Chapter 2's **Table 2-4** lists the same kind of numbers for the Llama models, under different names. The "Book's name" column shows the match.

**Overall size of the model**

| Row | What it means in plain words | Book's name (Table 2-4) | You use it in |
|---|---|---|---|
| `hidden_size` | How many numbers represent each token as it moves through the model: the model's "width". | Model dim | A.2 |
| `num_hidden_layers` | How many layers are stacked: the model's "depth". | # transformer blocks | A.2, A.4 |
| `intermediate_size` | How wide the MLP part of each layer is inside. Bigger means more parameters per layer. (In `gemma-moe` this is a small regular MLP that runs beside the experts.) | Feedforward dim | A.2 |
| `vocab_size` | How many different tokens the model knows. Each one gets its own row of `hidden_size` numbers in the embedding table. | Vocab size | A.2 |
| `tie_word_embeddings` | `True` means the model reuses its input embedding table as its output layer (the "unembedding" layer), instead of storing a second copy. Saves parameters. | (not in the table) | A.2 |

**Attention**

| Row | What it means in plain words | You use it in |
|---|---|---|
| `num_attention_heads` | Attention is split into several "heads" that each look for different patterns at the same time. This is how many. | A.2 |
| `num_key_value_heads` | How many sets of keys and values those heads share. Fewer than the number of heads saves memory (explained in A.4). | A.4 |
| `head_dim` | How many numbers each head works with. | A.4 |
| `layer types` | Not every layer uses the same kind of attention. Count of each kind, explained in the notes below the table. | A.4 |

**Mixture of experts (MoE models only)**

| Row | What it means in plain words | You use it in |
|---|---|---|
| `num_experts` | An MoE layer has many small MLPs called **experts** instead of one big one. This is how many each layer has. | A.1, A.2 |
| `num_experts_per_tok` / `top_k_experts` | How many experts a small "router" picks for each token. Only these do any work for that token. Same idea, different name in each model family. | A.1, A.2 |
| `moe_intermediate_size` | How wide each expert is inside (like `intermediate_size`, but for one small expert). | A.2 |

You don't need to memorize this. The big idea for this lab: **width x depth sets how many parameters a model stores, and for an MoE model, only the experts picked for each token do the work.** That gap between "stored" and "used" is what Part B measures.

A few more things to notice in the table:

- `qwen-moe` is *narrower* (`hidden_size` 2048) than `qwen-dense` (5120), yet it has more total parameters. The extra parameters live in its **experts**.
- The two MoE configs name the same idea differently: Qwen says `num_experts_per_tok`, Gemma says `top_k_experts`. Reading configs across model families always involves a little translation.
- **Most layers in these models don't keep a growing memory of every earlier token, so their KV cache (A.4) stays small.** That is all you need from this bullet for the lab. The details: most Qwen layers are `linear_attention` (a newer kind of layer, Gated DeltaNet, that keeps a fixed-size running summary of the text so far), and only one in four is ordinary `full_attention`. Gemma mixes `sliding_attention` (each token looks back at only the last 1,024 tokens) with a few `full_attention` layers. Chapter 2's **"Other model architectures"** discussion (RWKV, Mamba, Jamba) is about exactly this kind of design: keep the quality of attention while avoiding its cost on long inputs. You'll see the payoff in A.4.

**TODO A.1:** complete `moe_info(cfg)` so it works for both families. The parameter counter in A.2 needs these two numbers to tell stored experts from used ones.

In [ ]:
def moe_info(cfg):
    """Return (num_experts, experts_used_per_token). Dense models return (0, 0).

    Qwen MoE configs use 'num_experts' and 'num_experts_per_tok'.
    Gemma 4 configs use 'num_experts' and 'top_k_experts'.
    Example: moe_info(configs["qwen-moe"]) -> (256, 8)
    """
    # TODO: read the two numbers with cfg.get(...); use 0 when a key is missing
    num_experts = None
    per_token = None
    return num_experts, per_token


for a in configs:
    print(f"{a:11} experts (total, per token):", moe_info(configs[a]))

In [ ]:
ck.check("A1_moe_info_qwen", moe_info(configs["qwen-moe"]) == (256, 8), 2,
         "qwen-moe should give (256, 8). Use cfg.get('num_experts', 0) and cfg.get('num_experts_per_tok', ...).")
ck.check("A1_moe_info_gemma_and_dense",
         moe_info(configs["gemma-moe"]) == (128, 8) and moe_info(configs["qwen-dense"]) == (0, 0), 2,
         "Gemma uses the key 'top_k_experts'. Dense models have neither key, so return (0, 0).");

## A.2 Count the parameters

A transformer block has two big parts: **attention** and an **MLP** module (Chapter 2, **"Transformer architecture"**; its linear layers are also called feedforward layers). In most models the MLP holds the majority of the parameters, as you will see in your own counts. Modern models use a "gated" MLP with **three** weight matrices (gate, up, down), each of size `hidden_size x intermediate_size`, so:

`MLP parameters = 3 x hidden_size x intermediate_size`

(A weight **matrix** is a grid of parameters. One that is `hidden_size` by `intermediate_size` holds `hidden_size x intermediate_size` numbers, and the MLP has three of them.)

In a **mixture of experts** model, each expert is just a small MLP. A small **router** (itself a tiny layer) picks a few experts for each token. So a MoE layer has two numbers:
- **total** parameters = all experts (you must store all of them in memory)
- **active** parameters = only the experts used for one token (what you compute with)

The chapter's MoE example (section **"Model Size"**) is Mixtral 8x7B: 46.7B parameters in total, but only 12.9B active per token, so it runs at roughly the cost of a 12.9B model.

**TODO A.2:** write `mlp_params`, then use it once by hand: read `qwen-dense`'s `hidden_size` and `intermediate_size` off the A.1 table and compute the MLP parameters in **one** of its layers. The helper cell after that uses your function (and your `moe_info`) to count everything else for you: attention, embeddings, and the special layer types.

In [ ]:
def mlp_params(hidden_size, intermediate_size):
    """Parameters in one gated MLP (gate + up + down matrices, no biases).
    Example: mlp_params(10, 20) -> 600
    """
    # TODO: return the number of parameters, using the formula in the text above
    return None


print(mlp_params(10, 20))

In [ ]:
# TODO: type the two numbers you read off the A.1 table for qwen-dense (not configs[...] lookups).
dense_hidden = None
dense_intermediate = None
dense_mlp_one_layer = mlp_params(dense_hidden, dense_intermediate) if dense_hidden and dense_intermediate else None
if dense_mlp_one_layer:
    print(f"qwen-dense: {dense_mlp_one_layer:,} MLP parameters in one layer; x {configs['qwen-dense']['num_hidden_layers']} layers = "
          f"{dense_mlp_one_layer * configs['qwen-dense']['num_hidden_layers'] / 1e9:.1f}B parameters in MLPs")

Run the helper below (no changes needed). Read the comments if you are curious how attention layers are counted; you do not need to modify it.

In [ ]:
# ---- Helper: full parameter count from a config (just run) ----
def attention_params(cfg):
    """Parameters in all attention-type layers (approximate; ignores small norms and biases)."""
    h, total = cfg["hidden_size"], 0
    for lt in cfg.get("layer_types") or ["full_attention"] * cfg["num_hidden_layers"]:
        if lt == "linear_attention":          # Qwen's Gated DeltaNet layers
            kd = cfg["linear_num_key_heads"] * cfg["linear_key_head_dim"]
            vd = cfg["linear_num_value_heads"] * cfg["linear_value_head_dim"]
            total += (h * (2 * kd + 2 * vd) + h * 2 * cfg["linear_num_value_heads"]
                      + (2 * kd + vd) * cfg["linear_conv_kernel_dim"] + vd * h)
            continue
        nh = cfg["num_attention_heads"]
        if lt == "full_attention" and "global_head_dim" in cfg:   # Gemma 4 global layers
            hd = cfg["global_head_dim"]
            nkv = cfg.get("num_global_key_value_heads", cfg["num_key_value_heads"])
            kv_mats = 1 if cfg.get("attention_k_eq_v") else 2      # keys and values share weights
        else:                                                     # ordinary / sliding attention
            hd, nkv, kv_mats = cfg.get("head_dim", h // nh), cfg["num_key_value_heads"], 2
        q_out = nh * hd * (2 if cfg.get("attn_output_gate") else 1)
        total += h * q_out + kv_mats * h * nkv * hd + nh * hd * h      # Q (+gate), K, V, output
    return total


def mlp_layer_params(cfg):
    """(total, active) MLP parameters in ONE layer, using your mlp_params and moe_info."""
    h = cfg["hidden_size"]
    n_exp, k = moe_info(cfg)
    if not n_exp:
        p = mlp_params(h, cfg["intermediate_size"])
        return p, p
    expert = mlp_params(h, cfg["moe_intermediate_size"])
    always_on = 0
    if cfg.get("shared_expert_intermediate_size"):                       # Qwen: one shared expert
        always_on += mlp_params(h, cfg["shared_expert_intermediate_size"])
    if cfg.get("enable_moe_block") and cfg.get("intermediate_size"):     # Gemma 4: dense MLP beside experts
        always_on += mlp_params(h, cfg["intermediate_size"])
    router = h * n_exp
    return n_exp * expert + always_on + router, k * expert + always_on + router


def count_params(cfg):
    """Dictionary of parameter counts in billions."""
    L, h, V = cfg["num_hidden_layers"], cfg["hidden_size"], cfg["vocab_size"]
    emb = V * h * (1 if cfg.get("tie_word_embeddings") else 2)   # input embeddings (+ output head if untied)
    attn = attention_params(cfg)
    mlp_t, mlp_a = mlp_layer_params(cfg)
    norms = 2 * L * h
    return {"total_B": (emb + attn + L * mlp_t + norms) / 1e9,
            "active_B": (emb + attn + L * mlp_a + norms) / 1e9,
            "embedding_B": emb / 1e9, "attention_B": attn / 1e9, "mlp_total_B": L * mlp_t / 1e9}


try:
    params = pd.DataFrame({a: count_params(c) for a, c in configs.items()}).T.round(2)
    display(params)
except (TypeError, KeyError):
    params = None
    print("Complete the TODOs in A.1 and A.2 first (moe_info or mlp_params still returns None).")

**Reading the table:** one row per model, in billions of parameters. `total_B` is everything the model stores and `active_B` is what one token uses. `embedding_B`, `attention_B`, and `mlp_total_B` split the total into the embedding table (from Lab 01), all attention layers, and all MLP or expert layers. A few tiny pieces are left out of the split, so the three don't add up exactly to `total_B`.

Compare your counts with the names: 35B-**A3B** means about 35B total and 3B active; Gemma's model card says 25.2B total and 3.8B active. Your "active" number for `qwen-moe` will come out a little above 3B because we counted the whole embedding table as active. Looking up one row of a table is almost free, so many vendors leave embeddings out of the "active" number (though when the table is tied, it is also the output layer, and that part *is* used for every token). Parameter counts always depend on such counting conventions.

**TODO A.2 (continued):** compute the active fraction for each MoE model.

In [ ]:
# TODO: active_fraction = active_B / total_B for each MoE model (use the `params` table above)
active_fraction = {
    "qwen-moe": None,
    "gemma-moe": None,
}
print(active_fraction)

In [ ]:
ok_counts = params is not None
if ok_counts:
    P = params
    ck.record("qwen_moe_total_B", float(P.loc["qwen-moe", "total_B"]))
    ck.record("qwen_moe_active_B", float(P.loc["qwen-moe", "active_B"]))
    ck.record("qwen_dense_total_B", float(P.loc["qwen-dense", "total_B"]))
ck.check("A2_mlp_params", mlp_params(10, 20) == 600, 1, "mlp_params(10, 20) should be 600 (3 x 10 x 20).")
ck.check("A2_read_from_table", dense_mlp_one_layer == 3 * configs["qwen-dense"]["hidden_size"] * configs["qwen-dense"]["intermediate_size"], 1,
         "Read hidden_size and intermediate_size from the qwen-dense column of the A.1 table.")
ck.check("A2_dense_counts_plausible", ok_counts and 25 <= P.loc["qwen-dense", "total_B"] <= 29
         and 3.6 <= P.loc["qwen-small", "total_B"] <= 4.8, 2,
         "qwen-dense should land near 27B and qwen-small near 4B. Check mlp_params.")
ck.check("A2_moe_counts_plausible", ok_counts and 32 <= P.loc["qwen-moe", "total_B"] <= 37
         and 2.5 <= P.loc["qwen-moe", "active_B"] <= 4.2 and 23 <= P.loc["gemma-moe", "total_B"] <= 27, 2,
         "qwen-moe should be ~35B total / ~3-4B active; gemma-moe ~25B total. Check moe_info.")
af = active_fraction.get("qwen-moe")
ck.check("A2_active_fraction", af is not None and 0.05 <= af <= 0.15, 1,
         "Divide active_B by total_B for qwen-moe; it should be roughly 0.1.");

## A.3 How much memory do the weights need?

Chapter 2's **"Model Size"** section gives the rule of thumb: memory for the weights = number of parameters x bytes per parameter. A 7B model in 16-bit (2 bytes, since 8 bits make a byte) needs at least 14 GB just to hold its weights. Most models are published in **bf16**, a 16-bit number format designed for machine learning. **Quantization** (Chapter 7) stores weights in fewer bits, usually with a small loss in quality: 8-bit is 1 byte per parameter, 4-bit is half a byte.

**TODO A.3:** first predict, then compute. Without running anything yet: which model will need the most memory in bf16, and roughly how many GB? Write your guess in `my_guess` below. Then write `weight_gb` (use decimal gigabytes, 1 GB = 1e9 bytes) and see how close you were.

In [ ]:
my_guess = ""   # TODO (before you run the table): e.g. "qwen-??, about ?? GB"


def weight_gb(params_billion, bits):
    """Memory for the weights in GB. Example: weight_gb(7, 16) -> 14.0"""
    # TODO: bytes = number of parameters x bits / 8 (8 bits make a byte); then convert bytes to GB.
    #       Remember params_billion is in billions (x 1e9) and 1 GB = 1e9 bytes.
    return None


print(weight_gb(7, 16))

In [ ]:
mem = None
if params is not None and weight_gb(7, 16) is not None:
    mem = pd.DataFrame({
        "total_B": params["total_B"],
        "bf16_GB": params["total_B"].apply(lambda p: weight_gb(p, 16)),
        "8bit_GB": params["total_B"].apply(lambda p: weight_gb(p, 8)),
        "4bit_GB": params["total_B"].apply(lambda p: weight_gb(p, 4)),
    }).round(1)
    display(mem)
    print("Course cluster: each Mac Studio M3 Ultra has 96 GB of unified memory (about 72 GB usable by the GPU by default).")
    print("Free Colab T4: 16 GB of GPU memory, about 15 GB usable.")
    print("Your guess was:", my_guess or "(none)")
else:
    print("Complete the TODOs above first.")

In [ ]:
ck.check("A3_weight_gb", weight_gb(7, 16) == 14.0 and weight_gb(10, 4) == 5.0, 2,
         "weight_gb(7, 16) should be 14.0 and weight_gb(10, 4) should be 5.0.")
ck.check("A3_memory_table", mem is not None and 60 <= mem.loc["qwen-moe", "bf16_GB"] <= 75, 1,
         "The bf16 row for qwen-moe should be about 69 GB. Re-run the table cell after fixing weight_gb.")
ck.check("A3_made_a_guess", bool(my_guess.strip()), 1, "Write your prediction in my_guess (any honest guess earns the point).")
if mem is not None:
    ck.record("qwen_moe_8bit_GB", float(mem.loc["qwen-moe", "8bit_GB"]))
    ck.record("qwen_dense_4bit_GB", float(mem.loc["qwen-dense", "4bit_GB"]))

**Reading the table:** `total_B` is your parameter count from A.2, and the other three columns are the GB needed just to hold the weights at 16, 8, and 4 bits. Compare them with the two hardware lines printed under the table. The Mac Studios have **unified memory**: the CPU and GPU share one pool, which is why a Mac can hold models far bigger than a 16 GB graphics card.

Now connect this to the course cluster. The instructor runs `qwen-moe` at 8-bit on two Mac Studios (one copy on each; the gateway spreads requests between them, called **load balancing**) and puts `qwen-dense` at 4-bit together with `gemma-moe`, `qwen-small`, and the embedding and reranker models on the third box. Your table should roughly match the "weights in memory" column below. You will justify (or criticize) this layout in Q3.

**The course cluster at a glance** (you'll need this table for the analysis):

| Alias | Model | Precision on the cluster | Weights in memory | Runs on |
|---|---|---|---|---|
| `qwen-moe` | Qwen3.6-35B-A3B (MoE, 35B total / ~3B active) | 8-bit | ~36 GB | Mac #1 and Mac #2 (one copy each, load-balanced) |
| `qwen-dense` | Qwen3.8-27B (dense) | 4-bit | ~16 GB | Mac #3 |
| `gemma-moe` | Gemma 4 26B-A4B (MoE, ~25B total / ~4B active) | 4-bit | ~16.5 GB | Mac #3 |
| `qwen-small` | Qwen3.5-4B (dense) | 8-bit | ~5 GB | Mac #3 |
| `embed` | Qwen3-Embedding-0.6B | 16-bit | ~1.2 GB | Mac #3 |
| `rerank` | Qwen3-Reranker-0.6B | 4-bit | ~0.3 GB | Mac #3 |

Each Mac Studio M3 Ultra: 96 GB unified memory, about 72 GB usable by the GPU by default, about 819 GB/s memory bandwidth, up to 8 requests at once per Mac. Mac #3 holds about 40 GB of weights, the rest is room for KV cache. Weights sizes are approximate (the "~36 GB" includes the extra multi-token-prediction layers).

## A.4 The KV cache: memory that grows with the conversation

Weights are not the only memory cost. First, a little more about attention. In each attention layer, every token is turned into three short lists of numbers (vectors): a **query** (what this token is looking for), a **key** (what it can be matched on), and a **value** (the information it passes along). A new token compares its query with the keys of all earlier tokens, then takes a weighted mix of their values.

While generating, a transformer keeps the **keys and values** of every previous token for every attention layer, so it does not recompute them. This "KV cache" grows with context length (Chapter 9 returns to it in detail). For one token:

`KV bytes per token = 2 (K and V) x layers x kv_heads x head_dim x bytes_per_value`

Several attention heads share one set of keys and values ("grouped-query attention"), which is why the config lists far fewer `num_key_value_heads` than `num_attention_heads`.

**TODO A.4:** write `kv_bytes_per_token` for ordinary full-attention layers. Before you run the table after it, predict which model will need the **most** KV cache at 32K tokens (hint: look at `num_hidden_layers`, `num_key_value_heads`, and the `layer types` row of the A.1 table) and write it in `kv_guess`.

In [ ]:
kv_guess = ""   # TODO (before you run the KV table): one alias, e.g. "qwen-small"


def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, bytes_per_value=2):
    """KV-cache bytes for ONE token across n_layers full-attention layers (16-bit cache by default).
    Example: kv_bytes_per_token(32, 8, 128) -> 131072
    """
    # TODO: use the formula in the text above
    return None


print(kv_bytes_per_token(32, 8, 128))

The helper below applies your function per layer type. `linear_attention` layers keep a fixed-size state instead of a growing cache, and `sliding_attention` layers never keep more than 1,024 tokens. It also shows what the cache would be if *every* layer were full attention, for comparison.

In [ ]:
# ---- Helper: KV cache for a given context length (just run) ----
CONTEXT = 32_768   # the course gateway caps requests at about 32K tokens


def kv_cache_gb(cfg, context=CONTEXT, all_full=False):
    total = 0
    for lt in cfg.get("layer_types") or ["full_attention"] * cfg["num_hidden_layers"]:
        if lt == "linear_attention" and not all_full:
            continue                                        # fixed-size state, does not grow
        if lt == "full_attention" and "global_head_dim" in cfg:    # Gemma global layers
            per_tok = kv_bytes_per_token(1, cfg["num_global_key_value_heads"], cfg["global_head_dim"])
            if cfg.get("attention_k_eq_v"):
                per_tok //= 2                               # keys double as values
        else:
            per_tok = kv_bytes_per_token(1, cfg["num_key_value_heads"], cfg["head_dim"])
        tokens = min(context, cfg.get("sliding_window", context)) if (lt == "sliding_attention" and not all_full) else context
        total += per_tok * tokens
    return total / 1e9


kv = None
if kv_bytes_per_token(32, 8, 128) is not None:
    kv = pd.DataFrame({a: {
        "KV_KB_per_token(32K avg)": round(kv_cache_gb(c) * 1e9 / CONTEXT / 1024, 1),
        "KV_GB_at_32K": round(kv_cache_gb(c), 2),
        "KV_GB_if_all_full_attention": round(kv_cache_gb(c, all_full=True), 2),
    } for a, c in configs.items()}).T
    display(kv)
    print("Your guess:", kv_guess or "(none)", "| largest at 32K:", kv["KV_GB_at_32K"].idxmax())
else:
    print("Complete kv_bytes_per_token first.")

**Reading the table:** `KV_GB_at_32K` is the cache for one request that fills the whole 32K-token context. `KV_KB_per_token(32K avg)` is that divided by 32,768 tokens, in kilobytes: on average, what each extra token costs. Why an average? For these hybrid models the cost per token isn't constant (sliding layers stop growing at 1,024 tokens, linear layers never grow), so the per-token cost depends on the length. Using the average is fine for the estimates in Q3. `KV_GB_if_all_full_attention` is what the cache would be if every layer were ordinary full attention, so the gap between the last two columns is what the hybrid layer design saves. Every request in progress has its own KV cache, so ten students with long chats need about ten times these numbers.

In [ ]:
ck.check("A4_kv_formula", kv_bytes_per_token(32, 8, 128) == 131072, 2,
         "kv_bytes_per_token(32, 8, 128) should be 131072 = 2 x 32 x 8 x 128 x 2.")
ck.check("A4_kv_table", kv is not None and 0.5 <= kv.loc["qwen-moe", "KV_GB_at_32K"] <= 0.8
         and kv.loc["qwen-dense", "KV_GB_at_32K"] > kv.loc["qwen-moe", "KV_GB_at_32K"], 2,
         "qwen-moe should need about 0.67 GB of KV cache at 32K tokens, and qwen-dense more.")
ck.check("A4_made_a_guess", kv_guess.strip() in configs, 1, "Write one alias (for example \"qwen-dense\") in kv_guess; any honest guess earns the point.")
if kv is not None:
    ck.record("qwen_moe_kv_GB_32K", float(kv.loc["qwen-moe", "KV_GB_at_32K"]))
    ck.record("qwen_dense_kv_GB_32K", float(kv.loc["qwen-dense", "KV_GB_at_32K"]))

# Part B - Applied lab: does active-parameter count predict speed?

Generating text happens one token at a time (Chapter 2, **"Sampling"**, which you'll read next week). For every new token the hardware must read every *active* weight from memory once. On a single request, that memory read, not the math, is usually the bottleneck. Why? The chip can do the math for one token far faster than it can pull many gigabytes of weights out of memory, like a fast cook who spends most of the time waiting on a slow pantry. So speed is mostly decided by how many **bytes** must be read per token. **Memory bandwidth** is how many gigabytes per second the chip can read from its memory. So a rough upper limit is:

`tokens per second  <=  memory bandwidth / bytes of active weights`

An M3 Ultra has about **819 GB/s** of memory bandwidth (a T4 has about 320 GB/s). The cell below turns your Part A numbers into a "speed limit" for each model on the cluster. Real speeds will be lower (software overhead, other students' requests) and sometimes higher than you expect (`qwen-moe` uses multi-token prediction: it guesses a few tokens ahead and checks the guesses together, a form of speculative decoding that can produce more than one token per weight read; Chapter 9 explains it).

**Watch the precision.** The cluster stores `qwen-moe` at 8 bits (1 byte per weight) but `qwen-dense` at 4 bits (half a byte), so compare **bytes**, not parameters: about 3.5B active x 1 byte for `qwen-moe` versus about 27B x 0.5 byte for `qwen-dense`. The active parameters differ by about 8x, but the bytes read per token differ by only about 4x. The cell below prints the speedup that the bytes predict; Q2 asks you to compare your measurement with it.

In [ ]:
BANDWIDTH_GB_S = 819
QUANT_BITS = {"qwen-moe": 8, "qwen-dense": 4, "qwen-small": 8}   # how the cluster stores each model
SPEED_MODELS = [MODELS["main"], MODELS["dense"], MODELS["small"]]

bound = {}
if params is not None and weight_gb(1, 8) is not None:
    for m in SPEED_MODELS:
        active_gb = weight_gb(params.loc[m, "active_B"], QUANT_BITS[m])
        bound[m] = BANDWIDTH_GB_S / active_gb
        print(f"{m:11} active weights ~{active_gb:5.1f} GB  ->  upper bound ~{bound[m]:5.0f} tokens/s")
    print(f"\nPredicted qwen-moe vs qwen-dense speedup from bytes read: {bound[MODELS['main']] / bound[MODELS['dense']]:.1f}x")
else:
    print("Complete Part A first.")

## B.1 Measure

You will send the same five Larkspur prompts to three models, twice each (30 requests plus 3 warm-up requests, about 3-6 minutes; `qwen-dense` is the slow one). The requests are interleaved so that a busy moment on the cluster does not hit only one model, and there is a short pause between calls to stay well under the **rate limit** (your key may send at most 60 requests per minute).

In `measure()` below, `temperature=0.7` is a sampling setting (Lab 03 covers it), and `extra_body=NO_THINK` turns off Qwen's "thinking" mode, in which the model writes out its reasoning before the answer. We want to time only the answer.

We measure from the API response:
- **latency**: wall-clock seconds for the whole request
- **output tokens/sec**: `usage.completion_tokens / latency`. This slightly underestimates pure generation speed because latency includes reading the prompt (called **prefill**) and network time. Chapter 9 separates these properly with streaming (receiving the answer token by token as it is written); today's rough number is enough.

**TODO B.1:** finish `measure()` (two lines).

In [ ]:
PROMPTS = [
    "Write a 120-word email to a Larkspur client explaining why their laptop needs a BIOS update tonight.",
    "Explain to a non-technical office manager, in about 120 words, what multi-factor authentication is.",
    "List eight practical steps a help-desk technician should take when a user says the printer is offline.",
    "In about 120 words, describe what a managed IT services provider does for a 20-person law firm.",
    "Write a short, friendly internal note (about 120 words) reminding technicians to log their time on tickets.",
]


def measure(model, prompt, max_tokens=300):
    """Send one request and return a dict with latency and output tokens/sec."""
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.7, extra_body=NO_THINK)
    latency = time.perf_counter() - t0
    # TODO: get the number of generated tokens from resp.usage.completion_tokens
    out_tokens = None
    # TODO: output tokens per second = out_tokens / latency
    tps = None
    return {"model": model, "latency_s": latency, "out_tokens": out_tokens, "tps": tps,
            "text": resp.choices[0].message.content}


print(measure(MODELS["small"], "Say hello in five words.", max_tokens=20))

Now run the experiment (just run; it prints progress). The first three requests are **warm-up** calls that are not measured: the first request to a model can be slow while it loads. If a request fails, it is recorded and the loop continues.

In [ ]:
REPEATS = 2
rows = []
for m in SPEED_MODELS:                                    # warm-up: first call can include model loading
    try:
        measure(m, "Reply with OK.", max_tokens=5)
    except Exception as e:
        print("warm-up failed for", m, e)

for rep in range(REPEATS):
    for i, prompt in enumerate(PROMPTS):
        for m in SPEED_MODELS:
            try:
                r = measure(m, prompt)
                r.update(rep=rep, prompt_id=i, error="")
            except Exception as e:
                r = {"model": m, "rep": rep, "prompt_id": i, "error": str(e)[:200]}
            rows.append(r)
            tps_txt = f"{r['tps']:.1f}" if isinstance(r.get("tps"), (int, float)) else "?"
            print(f"rep {rep} prompt {i} {m:11} latency={r.get('latency_s', 0):5.1f}s  tok/s={tps_txt}")
            time.sleep(1.0)

df = pd.DataFrame(rows)
print("\nrequests:", len(df), " errors:", int((df["error"] != "").sum()))

## B.2 Summarize and plot

**TODO B.2:** make a `summary` table with one row per model and the **median** `latency_s`, `out_tokens`, and `tps`. (The median is the middle value; unlike the average, one unusually slow request can't drag it off.) One line with `groupby` does it: it groups the rows by model, like a pivot table in Excel. Then compute the MoE-vs-dense speedup.

In [ ]:
ok_rows = df[df["error"] == ""].copy() if "error" in df else df

# TODO: summary = ok_rows.groupby("model")[["latency_s", "out_tokens", "tps"]].median()
summary = None

# TODO: speedup = median tps of qwen-moe divided by median tps of qwen-dense (use summary.loc[...])
speedup_moe_vs_dense = None

if summary is not None:
    if bound:
        summary["upper_bound_tps"] = pd.Series(bound)
        summary["percent_of_bound"] = (100 * summary["tps"] / summary["upper_bound_tps"]).round(0)
    display(summary.round(2))
print("MoE vs dense speedup:", speedup_moe_vs_dense)

**Reading the summary:** one row per model; each value is the median over that model's 10 requests. `latency_s` is seconds for a whole request, `out_tokens` is how many tokens the model wrote, and `tps` is output tokens per second. `upper_bound_tps` is the bandwidth speed limit from the start of Part B, and `percent_of_bound` is how close the real speed came to it. The speedup is how many times faster `qwen-moe` writes than `qwen-dense`.

In [ ]:
# Just run: one answer from each model to the same prompt, so you can judge quality as well as speed (Q5).
if len(ok_rows) and "text" in ok_rows:
    for m in SPEED_MODELS:
        r = ok_rows[(ok_rows["model"] == m) & (ok_rows["prompt_id"] == 0)]
        if len(r):
            print(f"===== {m} =====\n{str(r.iloc[0]['text'])[:700]}\n")

In [ ]:
# ---- Plot (just run) ----
if summary is not None and params is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    order = [m for m in SPEED_MODELS if m in summary.index]
    axes[0].bar(order, [summary.loc[m, "tps"] for m in order], color="#4C72B0")
    axes[0].set_ylabel("median output tokens / second")
    axes[0].set_title("Measured generation speed")
    for m in order:
        x = params.loc[m, "active_B"]
        axes[1].scatter(x, summary.loc[m, "tps"], s=80, label=f"{m} measured")
        if m in bound:
            axes[1].scatter(x, bound[m], marker="x", s=80, color="gray")
        axes[1].annotate(m, (x, summary.loc[m, "tps"]), textcoords="offset points", xytext=(5, 5))
    axes[1].set_xlabel("active parameters (billions)")
    axes[1].set_ylabel("tokens / second")
    axes[1].set_title("Speed vs active parameters (x = bandwidth upper bound)")
    plt.tight_layout()
    plt.show()
else:
    print("Complete the TODOs above first.")

**Reading the plot:** the left chart is the measured speed of each model. The right chart places each model by its active parameters (across) and its speed (up): colored dots are your measurements and gray x marks are the bandwidth upper bound for the same model. If speed is mostly decided by active parameters, the dots fall as you move right. Remember the models are stored at different precisions, so the bytes read per token (what the gray x marks are based on) don't line up exactly with the parameters on the x-axis.

In [ ]:
n_ok = len(ok_rows)
ck.check("B1_measure_returns_tps", "tps" in ok_rows and ok_rows["tps"].notna().all() and n_ok > 0
         and (ok_rows["tps"] > 0).all(), 2,
         "measure() must fill out_tokens from resp.usage.completion_tokens and tps = out_tokens / latency.")
ck.check("B1_enough_requests", n_ok >= 27, 2,
         "At least 27 of 30 requests should succeed. Check the VPN, then re-run the experiment cell.")
ck.check("B2_summary_all_models", summary is not None and set(SPEED_MODELS) <= set(summary.index), 2,
         "summary needs one row per model: ok_rows.groupby('model')[[...]].median()")
ck.check("B2_speedup_computed", speedup_moe_vs_dense is not None and summary is not None
         and abs(speedup_moe_vs_dense - summary.loc[MODELS["main"], "tps"] / summary.loc[MODELS["dense"], "tps"]) < 1e-6, 2,
         "speedup = qwen-moe median tps / qwen-dense median tps (not the reverse), using summary.loc[...].")
if summary is not None:
    for m in SPEED_MODELS:
        if m in summary.index:
            ck.record(f"{m}_median_tps", float(summary.loc[m, "tps"]))
            ck.record(f"{m}_median_latency_s", float(summary.loc[m, "latency_s"]))
if speedup_moe_vs_dense is not None:
    ck.record("speedup_moe_vs_dense", float(speedup_moe_vs_dense))
    if speedup_moe_vs_dense <= 1.0:
        print("Note: qwen-moe was not faster in your run. That is a real result (busy cluster, short answers); report it and explain it in Q2.")

# Written analysis

Answer each question in the box under it (3-6 sentences each unless the question says otherwise). Use **your own numbers** from this notebook, and cite the chapter by section name where asked. Graded by the instructor (40 points).

**Q1. Total versus active parameters.** From your Part A table, how many parameters does `qwen-moe` store and how many does it use per token? What fraction is that? Using Chapter 2's discussion of sparse and mixture-of-experts models (section **"Model Size"**, including the Mixtral example), explain why a MoE model needs memory for its *total* parameters but costs compute for only its *active* parameters.

*Your answer:*

**Q2. Speed and memory bandwidth.** Report your measured median tokens/sec for the three models and the MoE-vs-dense speedup. Compare the speedup to the one the bytes predict (printed under the "upper bound" cell; remember the two models use different precisions). Give at least two concrete reasons your measurements differ from the bound.

*Your answer:*

**Q3. Hardware fit.** Three short parts (2-3 sentences each):
- **(a)** Using the cluster table in A.3 and your KV-cache table, add up Mac #3 (all five models' weights plus KV cache for, say, four 32K-token requests). Does it fit under about 72 GB? Show the sum.
- **(b)** Which of the four chat models could you run on a free Colab T4 (15 GB), at what precision, and roughly how much room would be left for KV cache (use the "32K avg" column)?
- **(c)** Why does the hybrid attention design (linear or sliding layers) matter for a cluster serving ~10 students at once?

*Your answer:*

**Q4. Scaling laws.** Chapter 2 (**"Scaling law: Building compute-optimal models"**) describes the Chinchilla finding of roughly 20 training tokens per parameter. Compute the Chinchilla-optimal number of training tokens for `qwen-small` (4B) and `qwen-dense` (27B). Qwen reported pretraining its Qwen3 models on about 36 trillion tokens. Why would a model developer train far past the "compute-optimal" point? Connect your answer to what you measured about inference speed and cost, and to the chapter's discussion of scaling bottlenecks (training data and electricity). Hint: look for what the chapter says about Llama and Sardana et al.

*Your answer:*

**Q5. A deployment recommendation for Larkspur.** Larkspur's help-desk assistant must answer technicians quickly during business hours, often with several requests at once. Based on your measurements and memory math, would you recommend a dense model or a MoE model of similar total size? State one trade-off that goes against your recommendation (think about what the chapter says about how model size relates to quality, and about memory). The side-by-side answers printed after B.2 are your quality evidence.

*Your answer:*

# AI-use reflection (10 points)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. AI assistants often state parameter counts or memory figures confidently. Did you get a number from an AI that disagreed with your own calculation from the config? Which one did you trust, and why?

*Your answer:*

## Submit
Run this last, save, upload the .ipynb.

In [ ]:
ck.report()